# Phase 2: test performance by chemical similarity

This notebook compares the frozen XGBoost, graph MPNN, and H1 (MPNN + descriptors) predictions on the scaffold test set. Four quantile bins are defined from each test molecule's maximum Morgan Tanimoto similarity to the **training** molecules. Duplicate cut points are collapsed so tied molecules remain together.

The bins use molecular features only. Labels are masked per endpoint when computing ROC AUC and PR AUC. This is a descriptive post-freeze analysis; it does not select a model or tune hyperparameters.

In [ ]:
from pathlib import Path
import sys

repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'dl').is_dir()), None)
if repo is None:
    raise RuntimeError('Open this notebook from the tox21-phase1 repository or a child directory')
sys.path.insert(0, str(repo))

from src.dl.tanimoto import load_scaffold_similarity, similarity_bins

y_test, max_similarity = load_scaffold_similarity(seed=0)
bin_ids, edges = similarity_bins(max_similarity, n_bins=4)
print(f'{len(y_test)} scaffold-test molecules')
print('Bin edges:', edges)
print('Molecules per bin:', [int((bin_ids == i).sum()) for i in range(max(bin_ids) + 1)])

## Compute endpoint metrics and export the figure

Each endpoint's missing labels are excluded by the shared evaluation metric. Metrics are averaged over valid endpoints inside each bin for each seed; the summary then reports the mean and sample SD across seeds. The raw endpoint-level CSV retains per-endpoint counts and metrics.

In [ ]:
from IPython.display import display
from src.config import RESULTS, SEEDS
from src.dl.tanimoto import run_tanimoto_analysis

raw, summary, missing = run_tanimoto_analysis(
    split='scaffold', seeds=SEEDS, n_bins=4,
    results_dir=RESULTS, output_dir=repo / 'report_phase2'
)
display(summary[['model', 'similarity_bin', 'n_molecules', 'pr_auc_mean', 'pr_auc_seed_sd', 'roc_auc_mean']])
if missing:
    print('Prediction artifacts not available:', *missing, sep='\n  ')
else:
    print('All three model families were included.')

## Reproducible outputs

Running the previous cell writes `report_phase2/figures/tanimoto.pdf`, a per-endpoint table, an aggregated table, and a LaTeX table ready for inclusion in the report. The bins are derived once from the scaffold split and reused unchanged across all model seeds.